# 04 — Error Analysis: LightGBM, Random Forest, XGBoost

`03_model_extension.ipynb` found LightGBM the clear leader (PR-AUC 0.245), with Random Forest and
XGBoost tied a step behind (0.229 each). Aggregate metrics can't say *where* each model succeeds
or fails, or whether the two runners-up are making the same mistakes as the leader or genuinely
different ones -- which is exactly what decides whether an ensemble is worth trying later. This
notebook does that: segment-level performance, confusion-matrix error types, cross-model error
overlap, feature-importance comparison, and a qualitative look at specific missed/caught clients.

Per the sequencing agreed with the team, this happens **before** any hyperparameter tuning --
it's cheap (reuses the same out-of-fold predictions as `03`, no new modelling ideas needed to
start), and its findings should decide what gets tuned and whether ensembling is worth trying,
rather than tuning blind.

## Imports and constants

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_FEATURES_PATH = PROCESSED_DIR / "df_training.csv"

RSEED = 42
N_SPLITS = 5
TOP_K_SHARE = 0.10

if not TRAINING_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Missing {TRAINING_FEATURES_PATH}. Run 01_eda.ipynb with "
        "SAVE_TRAINING_ARTIFACTS=true first."
    )

## Step 1 — Load the cleaned, client-level features

In [ ]:
df = pd.read_csv(TRAINING_FEATURES_PATH)
df["client_id"] = df["client_id"].astype("string")
df["target"] = df["target"].astype("int8")

expected_columns = {
    "client_id", "target",
    "invoice_count", "active_days", "mean_consumption", "zero_consumption_rate",
    "elec_share", "mean_invoice_gap_days", "backwards_index_rate", "meter_count",
    "mean_monthly_submission_index_delta", "backward_submission_rate",
    "large_mismatch_count", "reconciliation_gap_abs_mean",
    "client_catg", "disrict", "region",
}
missing_columns = sorted(expected_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"{TRAINING_FEATURES_PATH} is missing required baseline columns: {missing_columns}")

overview = pd.DataFrame({
    "rows": [len(df)], "unique_clients": [df["client_id"].nunique()], "fraud_rate": [df["target"].mean()],
})
display(overview)

## Step 2 — Preprocessing (identical to `02`/`03`)

This notebook uses the same finalized 15-input contract as the baseline and model extension: 12 numeric features plus the categorical codes `client_catg`, `disrict`, and `region`. Raw dates, tenure, identifiers, and investigation-related fields are not model inputs. Numeric values are median-imputed and scaled; categories are one-hot encoded inside the cross-validation pipeline.

In [ ]:
categorical_features = ["client_catg", "region", "disrict"]
numeric_features = [
    "invoice_count", "active_days", "mean_consumption", "zero_consumption_rate",
    "elec_share", "mean_invoice_gap_days", "backwards_index_rate", "meter_count",
    "mean_monthly_submission_index_delta", "backward_submission_rate",
    "large_mismatch_count", "reconciliation_gap_abs_mean",
]

model_features = numeric_features + categorical_features
X = df[model_features]
y = df["target"]
y_values = y.to_numpy()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RSEED)

## Step 3 — Recompute out-of-fold predictions for the three tree ensembles

Same model configurations as `03_model_extension.ipynb` (same `class_weight`/`scale_pos_weight`,
same seed, same CV) -- duplicated rather than imported so this notebook runs on its own. Only
the three tree ensembles are needed here; logistic regression, KNN, and Isolation Forest already
have their story from `03` and aren't the subject of this analysis.

In [ ]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

model_specs = {
    "lightgbm": LGBMClassifier(class_weight="balanced", random_state=RSEED, n_jobs=-1, verbosity=-1),
    "random_forest": RandomForestClassifier(class_weight="balanced", random_state=RSEED, n_jobs=-1),
    "xgboost": XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=RSEED, n_jobs=-1
    ),
}

oof_probas = {}
for name, model in model_specs.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", model)])
    oof_probas[name] = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")[:, 1]
    print(f"done: {name}")

model_names = list(model_specs.keys())

# Per-model "flagged as fraud" mask, using the same top-10% inspection-budget threshold as the
# "Confusion matrix @ top-10%" figures cited in the Results section below.
flags = {
    name: oof_probas[name] >= np.quantile(oof_probas[name], 1 - TOP_K_SHARE)
    for name in model_names
}

## Step 4 — Do the three models miss the same fraud, or different fraud?

The question that actually decides whether ensembling is worth trying later: for each fraud
client, count how many of the three models missed them (0 = every model caught them, 3 = every
model missed them). If misses cluster at 0 and 3, the models agree with each other and an
ensemble won't help much. If there's a lot of "missed by exactly 1 or 2," the models are making
different mistakes, and averaging/combining them could genuinely catch more fraud than any one
model alone.

In [ ]:
fraud_positions = np.flatnonzero(y_values == 1)
non_fraud_positions = np.flatnonzero(y_values == 0)

missed_by = pd.DataFrame(
    {name: ~flags[name][fraud_positions] for name in model_names}, index=fraud_positions
)
missed_count = missed_by.sum(axis=1)
print("Among all fraud clients, how many of the 3 models missed them:")
display(missed_count.value_counts().sort_index().rename("fraud_clients").to_frame())

falsely_flagged_by = pd.DataFrame(
    {name: flags[name][non_fraud_positions] for name in model_names}, index=non_fraud_positions
)
falsely_flagged_count = falsely_flagged_by.sum(axis=1)
print("\nAmong all non-fraud clients, how many of the 3 models falsely flagged them:")
display(falsely_flagged_count.value_counts().sort_index().rename("non_fraud_clients").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
missed_count.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#D62828")
axes[0].set(title="Fraud clients missed by how many models", xlabel="Models that missed them", ylabel="Clients")
falsely_flagged_count.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#33658A")
axes[1].set(title="Non-fraud clients falsely flagged by how many models", xlabel="Models that flagged them", ylabel="Clients")
plt.tight_layout()
plt.show()

### Precision and recall at the top-10% inspection threshold

The per-model `flags` from Step 3 are what turn a continuous score into an actual confusion
matrix at a capacity an inspection team could act on: **precision@10%** is the share of flagged
clients who are truly fraud (how well the team's limited inspection time would be spent),
**recall@10%** is the share of *all* fraud clients that flag catches (how much fraud gets found
at all, at that budget). This is the concrete, capacity-aware number Steps 1-3's PR-AUC/ROC-AUC
don't give directly -- those score the whole ranking, not the specific 10% an inspection team can
actually act on. This is the source of the "Confusion matrix @ top-10%" figures in the Results
section below.

In [ ]:
confusion_rows = {}
for name in model_names:
    predicted_positive = flags[name]
    true_positives = int((predicted_positive & (y_values == 1)).sum())
    false_positives = int((predicted_positive & (y_values == 0)).sum())
    false_negatives = int((~predicted_positive & (y_values == 1)).sum())
    true_negatives = int((~predicted_positive & (y_values == 0)).sum())
    confusion_rows[name] = {
        "flagged_clients": int(predicted_positive.sum()),
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "true_negatives": true_negatives,
        "precision_at_k": true_positives / (true_positives + false_positives),
        "recall_at_k": true_positives / (true_positives + false_negatives),
    }

confusion_at_k = pd.DataFrame(confusion_rows).T
print(f"Confusion matrix @ top-{TOP_K_SHARE:.0%} inspection threshold, per model:")
display(confusion_at_k)

In [ ]:
# `analysis` is the per-client frame Steps 5-11 slice by segment: `df` plus an
# `invoice_count_quintile` bucket (client_catg/region/disrict are already columns on `df`).
# It keeps `df`'s original row order, so positions here line up with `fraud_positions`,
# `non_fraud_positions`, and the per-model `flags` from Step 3.
analysis = df.copy()
analysis["invoice_count_quintile"] = pd.qcut(
    analysis["invoice_count"], q=5, labels=[f"Q{i}" for i in range(1, 6)], duplicates="drop"
)

### Recall@10% by segment

The confusion matrix above is one aggregate number per model; this breaks recall@10% down by
`invoice_count_quintile`, `client_catg`, `disrict`, and `region` for each model -- the same
top-10% flag, sliced by segment, is what actually answers "do the models miss the same fraud, or
different fraud, and where." A segment is only reported once it has at least 15 fraud clients, so
a single caught/missed client can't swing a rate to a misleading number. `segment_recall_table`
is written once and reused for all four columns, and reused again by Step 8 below (as
`quintile_table`) so there is one source of truth for segment-level recall in this notebook.

In [ ]:
def segment_recall_table(frame, column, minimum_fraud_cases=15):
    """One row per segment value of `column`, with each model's recall@top-10% (share of that
    segment's fraud clients its `flags` caught). Segments below `minimum_fraud_cases` fraud
    clients are dropped."""
    rows = []
    for segment_value, group in frame.groupby(column, observed=True, dropna=False):
        positions = group.index.to_numpy()
        fraud_mask = y_values[positions] == 1
        fraud_cases = int(fraud_mask.sum())
        if fraud_cases < minimum_fraud_cases:
            continue
        row = {"segment": segment_value, "clients": len(group), "fraud_cases": fraud_cases}
        for name in model_names:
            row[f"{name}_recall"] = flags[name][positions][fraud_mask].mean()
        rows.append(row)
    return pd.DataFrame(rows)


segment_tables = {
    column: segment_recall_table(analysis, column)
    for column in ["invoice_count_quintile", "client_catg", "disrict", "region"]
}
for column, table in segment_tables.items():
    print(f"Recall@10% by {column}:")
    display(table.sort_values("segment").reset_index(drop=True))

quintile_table = segment_tables["invoice_count_quintile"]  # Step 8 needs this one specifically

## Step 5 — Where the models agree and disagree on catching fraud

Step 4 counted *how many* models missed each fraud client; this section shows *which* clients
and *where*. Two views: a client-by-client scatter of each pair of models' scores (colored by
how many models missed that client), and a segment-level breakdown of the same agreement
categories -- then a dedicated close-up on the clients every model misses, since that's the
group we most need to understand.

In [ ]:
agreement_labels = {
    0: "caught by all 3", 1: "missed by 1 of 3", 2: "missed by 2 of 3", 3: "missed by all 3",
}
fraud_agreement = missed_count.map(agreement_labels)  # index = fraud_positions, from Step 6

palette = {
    "caught by all 3": "#2E7D32", "missed by 1 of 3": "#F4A300",
    "missed by 2 of 3": "#D62828", "missed by all 3": "#6A4C93",
}
agreement_order = list(palette.keys())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pairs = [("lightgbm", "random_forest"), ("lightgbm", "xgboost"), ("random_forest", "xgboost")]
for ax, (model_a, model_b) in zip(axes, pairs):
    for label in agreement_order:
        positions = fraud_agreement[fraud_agreement == label].index
        ax.scatter(oof_probas[model_a][positions], oof_probas[model_b][positions],
                   s=14, alpha=0.5, color=palette[label], label=label)
    ax.set(xlabel=f"{model_a} score", ylabel=f"{model_b} score",
           title=f"{model_a} vs {model_b}\n(fraud clients only)")
axes[0].legend(fontsize=7.5, loc="upper left", title="Agreement")
fig.suptitle("Every point is one fraud client -- colour shows how many models missed them")
plt.tight_layout()
plt.show()

### By segment

The same four agreement categories, now broken down by invoice-count quintile and by
`client_catg` -- do the disagreements and unanimous misses concentrate in particular segments,
or are they spread evenly?

In [ ]:
analysis["agreement"] = pd.NA
analysis.loc[fraud_agreement.index, "agreement"] = fraud_agreement.values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, column in zip(axes, ["invoice_count_quintile", "client_catg"]):
    counts = (
        analysis.dropna(subset=["agreement"])
        .groupby([column, "agreement"], observed=True).size()
        .unstack("agreement").reindex(columns=agreement_order)
    )
    counts.plot(kind="bar", stacked=True, ax=ax, color=[palette[c] for c in agreement_order])
    ax.set(title=f"Fraud clients by {column}\nand model agreement", ylabel="Fraud clients")
    ax.legend(fontsize=7, title="Agreement")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### A closer look at the fraud clients every model misses

Not mixed in with the other categories this time -- just the "missed by all 3" group on its
own, to see concretely where these clients concentrate.

In [ ]:
missed_by_all = analysis[analysis["agreement"] == "missed by all 3"]
print(f"{len(missed_by_all)} fraud clients missed by all 3 models -- where do they concentrate?")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, column in zip(axes, ["invoice_count_quintile", "client_catg", "disrict"]):
    missed_by_all[column].value_counts(normalize=True).sort_index().plot(
        kind="bar", ax=ax, color="#6A4C93"
    )
    ax.set(title=f"Missed-by-all-3 fraud clients\nby {column}", ylabel="Share of missed-by-all-3 clients")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## Step 6 — Feature importance comparison

Each model uses a different importance metric internally (LightGBM: split count, XGBoost: gain,
Random Forest: impurity decrease) -- these are **not comparable in magnitude across models**,
only within each model's own ranking. Shown side by side to see whether the three models are
relying on similar signals or genuinely different ones.

In [ ]:
fitted_pipelines = {}
for name, model in model_specs.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", model)])
    pipeline.fit(X, y)
    fitted_pipelines[name] = pipeline

feature_names = fitted_pipelines["lightgbm"].named_steps["preprocess"].get_feature_names_out()
feature_names = [name.split("__", 1)[-1] for name in feature_names]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for axis, name in zip(axes, model_names):
    importances = fitted_pipelines[name].named_steps["model"].feature_importances_
    top = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(12)
    top.iloc[::-1].plot(kind="barh", ax=axis, color="#33658A")
    axis.set(title=f"{name}: top 12 features (own scale)")
fig.tight_layout()
plt.show()

## Step 7 — A qualitative look at specific clients

A handful of fraud clients every model missed, and a handful every model caught -- not a
statistic, just a sanity-check read on what "hard" and "easy" actually look like here.

In [ ]:
inspect_columns = [
    "client_id", "client_catg", "region", "disrict", "invoice_count", "active_days",
    "mean_consumption", "backwards_index_rate", "reconciliation_gap_abs_mean",
]

hardest_positions = missed_count[missed_count == len(model_names)].index
print(f"Fraud clients missed by all {len(model_names)} models: {len(hardest_positions)}")
display(analysis.iloc[hardest_positions][inspect_columns].head(8))

easiest_positions = missed_count[missed_count == 0].index
print(f"\nFraud clients caught by all {len(model_names)} models: {len(easiest_positions)}")
display(analysis.iloc[easiest_positions][inspect_columns].head(8))

## Step 8 — How much of the 43.5% miss rate is the known short-history story, versus something else?

Step 7 found that 3,293 fraud clients (43.5% of all fraud) are missed by every model. Step 4's
segment breakdown already showed the shortest invoice-count quintile collapsing to 2.3-6.2%
recall -- plausibly a large share of that 3,293 is just the observation-window blind spot
restated. This step separates the two: which quintile bin(s) are actually collapsed (data-driven,
not assumed), what fraction of the missed-by-all group they explain, and -- for the *rest* of the
missed group (long-history clients still missed by every model) -- how far below each model's own
top-10% threshold they actually score. A client just under the line is a capacity/threshold
problem; a client scored low with confidence is a features problem.

In [ ]:
quintile_recall_cols = [f"{name}_recall" for name in model_names]
quintile_by_segment = quintile_table.sort_values("segment").reset_index(drop=True)
quintile_by_segment["max_recall"] = quintile_by_segment[quintile_recall_cols].max(axis=1)
display(quintile_by_segment[["segment", "clients", "fraud_cases"] + quintile_recall_cols + ["max_recall"]])

COLLAPSED_RECALL_THRESHOLD = 0.10  # a quintile bin counts as "already known short-history blind spot"
                                     # if every model's recall there is below this
collapsed_bins = set(
    quintile_by_segment.loc[quintile_by_segment["max_recall"] < COLLAPSED_RECALL_THRESHOLD, "segment"]
)
print(f"Quintile bin(s) already explained by the known short-history blind spot "
      f"(max recall across models < {COLLAPSED_RECALL_THRESHOLD:.0%}): {collapsed_bins}")

analysis["history_bucket"] = np.where(
    analysis["invoice_count_quintile"].isin(collapsed_bins),
    "short history (explained)",
    "long history (unexplained)",
)

missed_by_all = missed_by_all.copy()
missed_by_all["history_bucket"] = analysis.loc[missed_by_all.index, "history_bucket"]

missed_split = missed_by_all["history_bucket"].value_counts()
missed_split_pct = (missed_split / len(missed_by_all) * 100).round(1)
print(f"\n{len(missed_by_all)} fraud clients missed by all 3 models, split by history bucket:")
display(pd.DataFrame({"clients": missed_split, "share_of_missed_%": missed_split_pct}))

long_history_fraud = analysis[(analysis["target"] == 1) & (analysis["history_bucket"] == "long history (unexplained)")]
long_history_missed_rate = (long_history_fraud["agreement"] == "missed by all 3").mean()
print(
    f"\nAmong {len(long_history_fraud):,} long-history fraud clients, "
    f"{long_history_missed_rate:.1%} are still missed by every model -- "
    "the residual puzzle once the known short-history blind spot is accounted for."
)

bucket_colors = {"short history (explained)": "#6A4C93", "long history (unexplained)": "#D62828"}
fig, ax = plt.subplots(figsize=(6, 4.5))
missed_split.plot(kind="bar", ax=ax, color=[bucket_colors[label] for label in missed_split.index])
for i, (label, count) in enumerate(missed_split.items()):
    ax.text(i, count, f"{count}\n({missed_split_pct[label]}%)", ha="center", va="bottom")
ax.set(title="Missed-by-all-3 fraud clients, split by history bucket", ylabel="Fraud clients")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## Step 9 — Is the missed-by-all group hiding in the imputed feature gap?

4,270 train clients have no usable positive-day meter transition, so
`mean_monthly_submission_index_delta` and `backward_submission_rate` are missing for them --
the model pipeline silently median-imputes those clients at fit time. If missed-by-all-3 fraud
clients disproportionately fall in this group, the model isn't just weak on them, it's scoring
them as generically average by construction. The key check is whether this holds up even among
*long-history* clients (an independent blind spot) or is fully explained by short history (the
same story restated).

In [ ]:
analysis["missing_submission_features"] = analysis["mean_monthly_submission_index_delta"].isna()
assert (analysis["missing_submission_features"] == analysis["backward_submission_rate"].isna()).all(), \
    "Expected mean_monthly_submission_index_delta and backward_submission_rate to share one missingness mask"

non_fraud_missing_rate = analysis.loc[analysis["target"] == 0, "missing_submission_features"].mean()
missing_by_agreement = (
    analysis.loc[analysis["target"] == 1]
    .groupby("agreement")["missing_submission_features"].mean()
    .reindex(agreement_order)
)

print(f"Clients missing the meter-transition-based features (train set): "
      f"{int(analysis['missing_submission_features'].sum()):,}")
print(f"Non-fraud missing rate (baseline): {non_fraud_missing_rate:.1%}\n")
display((missing_by_agreement * 100).round(1).rename("missing_rate_%").to_frame())

fig, ax = plt.subplots(figsize=(7, 4.5))
(missing_by_agreement * 100).plot(kind="bar", ax=ax, color=[palette[c] for c in agreement_order])
ax.axhline(non_fraud_missing_rate * 100, color="black", linestyle="--", label="non-fraud baseline")
ax.set(title="Missing meter-transition features, by model agreement", ylabel="Missing rate (%)")
ax.legend()
plt.xticks(rotation=12)
plt.tight_layout()
plt.show()

missing_by_history_and_agreement = (
    analysis.loc[analysis["target"] == 1]
    .groupby(["history_bucket", "agreement"])["missing_submission_features"].mean()
    .unstack("agreement").reindex(columns=agreement_order)
)
print("\nMissing-feature rate by history bucket and model agreement (fraud clients only):")
display((missing_by_history_and_agreement * 100).round(1))

## Step 10 — What do the genuinely puzzling clients' actual invoice histories look like?

Steps 8-9 characterize the "missed despite long history" group statistically. This step looks
at a handful of them directly: their real invoice sequences from `data/parquet/invoice_train.parquet`
(raw, per-invoice, never aggregated away) -- a handful of illustrative examples from the group
Steps 8-9 characterize as still puzzling.

> **Leakage note:** `counter_statue` and `reading_remarque` are shown below for these clients'
> records. Both are excluded from every trained model in this project (`01_eda.ipynb` Step 7 --
> their "unusual code" rate jumps sharply at each client's *last* invoice, more so for fraud
> clients, suggesting a later investigation/account-closure event rather than an independent
> predictor). They appear here **only** for human, qualitative inspection of these already-identified
> clients -- no model in this notebook, or anywhere in this project, trains on them.

n≈6 is illustrative, not statistical evidence -- treat this as narrative color for the
presentation, not a finding to act on directly.

In [ ]:
puzzling_missed = missed_by_all[missed_by_all["history_bucket"] == "long history (unexplained)"]
print(f"{len(puzzling_missed)} clients missed by all 3 models despite a long invoice history -- "
      "the genuinely puzzling subset examined below.")

N_EXAMPLES = 6
selected_client_ids = puzzling_missed["client_id"].astype("string").head(N_EXAMPLES).tolist()
print(f"\nSelected {len(selected_client_ids)} 'missed despite long history' clients for a qualitative look.")

invoices = pd.read_parquet(DATA_DIR / "parquet" / "invoice_train.parquet")
invoices["client_id"] = invoices["client_id"].astype("string")
client_invoices = invoices[invoices["client_id"].isin(selected_client_ids)].copy()
client_invoices["invoice_date"] = pd.to_datetime(client_invoices["invoice_date"])
client_invoices["index_delta"] = client_invoices["new_index"] - client_invoices["old_index"]
client_invoices = client_invoices.sort_values(["client_id", "invoice_date"])
print(f"Loaded {len(client_invoices):,} raw invoice rows for the {len(selected_client_ids)} selected clients.")

inspect_invoice_columns = [
    "invoice_date", "tarif_type", "counter_type",
    "consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4",
    "old_index", "new_index", "index_delta", "months_number",
]

for client_id in selected_client_ids:
    client_rows = client_invoices[client_invoices["client_id"] == client_id]
    print(f"\n=== client {client_id} ({len(client_rows)} invoices) ===")
    display(client_rows[inspect_invoice_columns])

    for flag_col in ["counter_statue", "reading_remarque"]:
        modal_code = client_rows[flag_col].mode().iloc[0]
        last_codes = client_rows[flag_col].tail(3)
        anomalous = last_codes[last_codes != modal_code]
        if len(anomalous):
            print(f"  {flag_col}: last 3 codes {last_codes.tolist()} vs. this client's own modal code "
                  f"{modal_code!r} -- {len(anomalous)} of the last 3 differ from their own norm.")
        else:
            print(f"  {flag_col}: last 3 codes match this client's own modal code ({modal_code!r}) -- no anomaly.")

### Consumption trajectories for the same clients

A visual complement to the tables above -- does a "quiet" fraud client's consumption actually
look flat/stable over time, consistent with the mean/median-divergence hypothesis from Step 10?

In [ ]:
client_invoices["total_consumption"] = client_invoices[
    ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]
].sum(axis=1)

n_selected = len(selected_client_ids)
ncols = min(3, n_selected) if n_selected else 1
nrows = int(np.ceil(n_selected / ncols)) if n_selected else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)
for ax, client_id in zip(axes.flat, selected_client_ids):
    client_rows = client_invoices[client_invoices["client_id"] == client_id]
    ax.plot(client_rows["invoice_date"], client_rows["total_consumption"], marker="o", markersize=3, color="#33658A")
    ax.set(title=f"{client_id}")
    ax.tick_params(axis="x", rotation=30, labelsize=7)
for ax in axes.flat[n_selected:]:
    ax.axis("off")
fig.suptitle("Consumption over time for the selected 'missed despite long history' clients")
plt.tight_layout()
plt.show()

### Supporting calculation — current features after controlling for history

The earlier version compared postponed features that are no longer exported in `df_training.csv`. This replacement uses the finalized 15-feature contract only. It compares long-history fraud clients missed by all three models with long-history non-fraud clients, so the result is not simply measuring the known short-history effect.


In [ ]:
# Compare the current numeric inputs after restricting both groups to long-history clients.
controlled_numeric_features = [
    column for column in numeric_features
    if column in analysis.columns and column not in {"invoice_count", "active_days"}
]
long_history = analysis[analysis["history_bucket"] == "long history (unexplained)"]
long_history_non_fraud = long_history[long_history["target"] == 0]
long_history_missed = long_history[
    (long_history["target"] == 1)
    & (long_history["agreement"] == "missed by all 3")
]

controlled_rows = []
for feature in controlled_numeric_features:
    reference = long_history_non_fraud[feature].astype(float)
    missed = long_history_missed[feature].astype(float)
    reference_iqr = reference.quantile(0.75) - reference.quantile(0.25)
    if reference_iqr == 0 or pd.isna(reference_iqr):
        effect_size = np.nan
    else:
        effect_size = (missed.median() - reference.median()) / reference_iqr
    controlled_rows.append({
        "feature": feature,
        "long_history_non_fraud_median": reference.median(),
        "long_history_missed_median": missed.median(),
        "effect_size": effect_size,
    })

controlled_result = (
    pd.DataFrame(controlled_rows)
    .set_index("feature")
    .sort_values("effect_size", key=lambda values: values.abs(), ascending=False)
)
print(
    f"Compared {len(long_history_missed):,} long-history missed fraud clients with "
    f"{len(long_history_non_fraud):,} long-history non-fraud clients."
)
display(controlled_result.head(8))


## Step 11 — Synthesis table

One compact table pulling together Steps 8-9's headline numbers, sized to paste into
`slides.md`'s error-analysis section later (that pasting is out of scope for this notebook --
see the follow-ups note at the end). Cut this step first if time is short; Steps 8-10 already
answer "what makes them different" on their own.

In [ ]:
total_fraud_clients = len(fraud_positions)

synthesis_rows = [
    ("Fraud clients missed by all 3 models",
     f"{len(missed_by_all):,} ({len(missed_by_all) / total_fraud_clients:.1%} of all fraud clients)"),
    ("  - short history (explained)",
     f"{missed_split.get('short history (explained)', 0):,} "
     f"({missed_split_pct.get('short history (explained)', 0)}% of the missed group)"),
    ("  - long history (unexplained)",
     f"{missed_split.get('long history (unexplained)', 0):,} "
     f"({missed_split_pct.get('long history (unexplained)', 0)}% of the missed group)"),
    ("Long-history fraud clients still missed by every model",
     f"{long_history_missed_rate:.1%} of {len(long_history_fraud):,} long-history fraud clients"),
    ("Strongest current feature difference after controlling for history",
     f"{controlled_result.index[0]} (effect size {controlled_result['effect_size'].iloc[0]:.2f})"),
    ("Missing meter-transition features: missed-by-all-3 vs. non-fraud baseline",
     f"{missing_by_agreement.get('missed by all 3', float('nan')):.1%} vs. {non_fraud_missing_rate:.1%}"),
]
synthesis_table = pd.DataFrame(synthesis_rows, columns=["Finding", "Value"])

# Plain display() truncates long strings at display.max_colwidth (default 50 chars) and adds a
# noisy numeric index. DataFrame.style needs the optional jinja2 package, which isn't part of
# this project's dependencies -- to_html() is pandas-core-only and avoids that, so it renders
# everywhere this notebook runs without adding a new dependency for one table.
from IPython.display import HTML

with pd.option_context("display.max_colwidth", None):
    synthesis_html = synthesis_table.to_html(index=False, justify="left", escape=False, classes="synthesis-table")

display(HTML(
    "<style>table.synthesis-table td, table.synthesis-table th "
    "{ text-align: left; padding: 6px 14px; }</style>" + synthesis_html
))
print(
    "\nSized to paste into slides.md's error-analysis section -- not done automatically "
    "(see the follow-ups note at the end of this notebook)."
)

## Results

> All figures below reflect a full, fresh run of every cell in this notebook (Steps 1-11) against
> the current `df_training.csv` -- not carried over from an earlier version of this notebook.
> `03_model_extension.ipynb`'s own PR-AUC ranking is cited for context only and has not been
> independently rerun here.

- **Confusion matrix @ top-10%**: LightGBM leads (precision 24.5%, recall 43.8%), XGBoost a step
  behind (23.8% / 42.7%), Random Forest close behind that (23.7% / 42.5%) -- consistent with
  `03_model_extension.ipynb`'s ranking.
- **`client_catg`**: all three models do noticeably *better* than their overall average on
  category 51 (the ~16.9%-fraud-rate segment) -- LightGBM 73.5% recall there vs. 43.8% overall --
  and noticeably *worse* on category 12 (recall 13.0-21.7%, but only 92 fraud cases, so read this
  cautiously rather than as a solid finding).
- **`disrict`**: district 63 is easier for every model (recall 55.7-58.7%) than the other three
  (33.1-45.0%), despite a mid-range fraud rate (6.5%) -- not the obvious "highest-fraud-rate" segment.
- **`region`**: real segment-robustness gap. Region 311 is easy for every model (recall
  65.5-69.3%), while regions 104 and 107 -- both with substantial fraud volume (720 and 658
  cases) -- are hard for every model (recall 22.2-29.9%). A strong overall number is hiding a
  meaningfully weaker region.
- **Invoice-count quintile -- the headline finding.** Recall for the shortest-history clients
  (Q1, 258 fraud cases) collapses to **2.3% (LightGBM), 5.4% (Random Forest), 6.2% (XGBoost)** --
  essentially none of the fraud in this group is caught, by any model -- compared to 51-54% for
  the longest-history quintile. This is the observation-window bias the EDA flagged as a labeling
  risk, now confirmed as a real, severe *model* blind spot, not just a curiosity about the data.
- **Model agreement is high**: of 7,566 fraud clients, 2,229 (29.5%) are caught by all three
  models and 3,293 (43.5%) are missed by all three -- 73.0% of fraud clients get the *same*
  verdict from every model. Only 2,044 (27.0%) see any disagreement. False positives follow a
  similar pattern: 111,866 non-fraud clients (87.4%) are correctly left alone by all three, and
  16,061 (12.6%) are flagged by at least one.
- **Feature importance**: LightGBM and Random Forest substantially agree on what matters
  (`active_days`, `mean_consumption`, `mean_invoice_gap_days`, `meter_count`, and
  `mean_monthly_submission_index_delta` dominate both top lists). **XGBoost is notably
  different** -- its top features are `meter_count` plus one-hot geographic codes (`disrict_69`,
  `region_101`, `region_103`, `region_301`, `region_311`), a mechanism the other two barely use
  at all.
- **The "missing-feature-gap" hypothesis (Step 9) doesn't hold up**: missed-by-all-3 fraud
  clients are missing the meter-transition features (`mean_monthly_submission_index_delta` /
  `backward_submission_rate`) at about the same rate as everyone else -- 1.6%, actually *below*
  the 4.1% non-fraud baseline. Whatever makes these ~3,061 long-history clients hard, it isn't
  captured by that particular imputation flag.
- **Step 10's controlled comparison finds only modest signal**: long-history fraud clients missed
  by every model invoice slightly more often than typical long-history non-fraud clients
  (`mean_invoice_gap_days` effect size -0.20 IQR) and have marginally more zero-consumption
  spells (+0.16 IQR) -- real differences, but small ones. Most of this group's feature values
  look unremarkable next to genuine non-fraud clients, which is exactly why the current
  15-feature contract can't separate them.

- **Step 5's scatter plots make the agreement finding visually concrete**: fraud clients form
  a clear diagonal band in every pairwise score comparison -- green ("caught by all 3") sits in
  the top-right (all scores high), purple ("missed by all 3") sits in the bottom-left (all
  scores low), and the orange/red disagreement cases fill the off-diagonal band where one model
  scores a client noticeably higher than another.
- **A nuance the segment charts surface that's easy to miss**: the *rate* of being missed is
  worst for short-history clients, but the *absolute number* of always-missed fraud clients is
  concentrated more in the longer-history quintiles simply because far more fraud cases exist
  there in total. Both are true at once: short-history clients are proportionally the model's
  biggest blind spot, but longer-history clients still account for more of the raw number of
  missed fraud cases. Worth stating both ways in the presentation rather than picking one.
- `client_catg`'s by-agreement chart is a good visual reminder of the sample-size caveat already
  noted: categories 12 and 51 are so small next to category 11 that their bars are barely
  visible in absolute terms -- read any category-12/51-specific finding as suggestive, not solid.

## Interpretation

- The high agreement between models (73% identical verdicts) explains why LightGBM, Random
  Forest, and XGBoost score similarly in aggregate: they're mostly finding the *same* fraud and
  missing the *same* fraud, not exploring genuinely different parts of the problem. The 3,293
  fraud clients missed by every model are the dataset's genuinely hard cases with the current
  features -- no amount of model-swapping alone is likely to catch them.
- The invoice-count collapse is the single most important finding here. A recall number like
  44% sounds usable for an inspection team, but that average is hiding a near-total blind spot
  for the newest, least-observed clients -- exactly the clients a real fraud program would most
  want early signal on. Reporting one aggregate recall number without this caveat would overstate
  the model's real-world usefulness.
- XGBoost's reliance on `disrict`/`region` one-hot codes is worth flagging as a fairness question,
  not just a technical curiosity: the EDA already cautioned that geographic fraud-rate patterns
  could partly reflect *where inspections have historically happened* rather than true fraud
  propensity (associations, not causes). XGBoost is structurally more exposed to that risk than
  LightGBM/Random Forest, which barely use geography at all -- a real consideration for model
  choice beyond raw PR-AUC.
- Steps 9-10 rule out the two most obvious "hidden reason" explanations for the missed-by-all-3
  group: it isn't concentrated in the imputed-feature gap, and no single current feature
  separates it strongly from ordinary non-fraud clients. Combined with the quintile finding, the
  honest read is that 41.9% of long-history fraud clients are missed not because of one
  identifiable gap, but because the current 15-feature contract simply doesn't carry a strong
  enough signal for them -- a features problem, not a modeling-effort problem.

### Next steps

- **Treat the invoice-count blind spot as the priority**, ahead of hyperparameter tuning: tuning
  a model's settings won't fix a missing-signal problem for short-history clients. Worth trying
  relative/normalized feature versions (e.g., consumption or gap features expressed per-invoice
  or per-active-day rather than as raw sums) that don't structurally require a long history to
  be informative, and stating the limitation explicitly in the presentation regardless.
- **Look closer at regions 104 and 107** before finalizing -- substantial fraud volume, poor
  recall across every model, worth understanding whether that's a genuine harder fraud pattern or
  a data/feature gap specific to those regions.
- **Try a soft (probability-averaging) ensemble of LightGBM + Random Forest** -- they share most
  of their top features and agree most often, so an ensemble is more about smoothing noise than
  combining genuinely different signals; test it against each individual model via the same CV
  scheme before adopting it. An "OR"-style union is not recommended as-is: it would recover 956
  of the 2,044 disagreement cases (4,273 fraud clients caught vs. LightGBM alone's 3,317) but
  raises false positives from LightGBM's 10,233 to a union total of 16,061 -- about 1.6x, not the
  free lunch a stale earlier estimate suggested.
- **Now proceed to hyperparameter tuning of LightGBM** (the confirmed single leader from `03`),
  informed by the above -- and re-check the invoice-count quintile breakdown afterward to confirm
  tuning didn't just improve the easy majority while leaving the short-history blind spot as bad
  as before.
- **`client_catg == 12`'s weak recall** is a low-confidence finding (92 fraud cases) -- flag it,
  don't act on it without more data.